# Module 8: Forecast Orchestration & Pipelines

This notebook demonstrates running the batch forecasting pipeline and inspecting its outputs.

You can run the pipeline either:
- from a terminal (recommended)
- from this notebook via `subprocess`


In [4]:
from pathlib import Path
import json
import pandas as pd
import subprocess

print('Ready')


Ready


## Run the pipeline

If you already ran it from terminal, skip this cell.

Otherwise, this runs:

- `python pipelines/forecasting_pipeline.py --config config/pipeline.yaml`


In [9]:
# Run the pipeline
# Run with repo root as working directory so imports and relative paths resolve consistently.
cmd = ['python', 'pipelines/forecasting_pipeline.py', '--config', 'config/pipeline.yaml']
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, cwd='..', capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')


Running: python pipelines/forecasting_pipeline.py --config config/pipeline.yaml
✓ Forecast pipeline complete
Forecasts: outputs\forecasts\20251217T082353Z_sku_forecasts.csv
Report: outputs\reports\20251217T082353Z_pipeline_report.json

c:\Users\aloag\personal-study\ecommerce-forecasting\pipelines\forecasting_pipeline.py:122: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")



## Inspect outputs

We’ll load the most recent pipeline report and its forecast CSV.


In [10]:
reports_dir = Path('../outputs/reports')
forecast_dir = Path('../outputs/forecasts')

report_files = sorted(reports_dir.glob('*_pipeline_report.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if not report_files:
    raise FileNotFoundError('No pipeline reports found in outputs/reports')

latest_report = report_files[0]
print('Latest report:', latest_report.name)
report = json.loads(latest_report.read_text(encoding='utf-8'))
report


Latest report: 20251217T082353Z_pipeline_report.json


{'run_id': '20251217T082353Z',
 'config': {'raw_path': 'data/raw/sample_sales.csv',
  'horizon_days': 14,
  'cutoff_mode': 'max_minus_horizon',
  'cutoff_date': None,
  'method': 'moving_average',
  'ma_window': 7,
  'ses_alpha': 0.3,
  'forecasts_dir': 'outputs/forecasts',
  'reports_dir': 'outputs/reports'},
 'data': {'raw_path': 'data\\raw\\sample_sales.csv',
  'rows': 365000,
  'date_min': '2023-12-18 00:00:00',
  'date_max': '2025-12-16 00:00:00',
  'unique_skus': 500},
 'forecast': {'cutoff': '2025-12-02 00:00:00',
  'horizon_days': 14,
  'method': 'moving_average',
  'rows': 7000,
  'output': 'outputs\\forecasts\\20251217T082353Z_sku_forecasts.csv'}}

In [11]:
forecast_path = Path(report['forecast']['output'])
if not forecast_path.is_absolute():
    # report stores repo-relative paths
    forecast_path = Path('..') / forecast_path

print('Forecast file:', forecast_path)
fc = pd.read_csv(forecast_path, parse_dates=['date'])
print('Shape:', fc.shape)
fc.head()


Forecast file: ..\outputs\forecasts\20251217T082353Z_sku_forecasts.csv
Shape: (7000, 6)


,sku_id,date,y_pred,method,category,subcategory
0,SKU001,2025-12-03,26.428571,moving_average,Electronics,Accessories
1,SKU001,2025-12-04,26.428571,moving_average,Electronics,Accessories
2,SKU001,2025-12-05,26.428571,moving_average,Electronics,Accessories
3,SKU001,2025-12-06,26.428571,moving_average,Electronics,Accessories
4,SKU001,2025-12-07,26.428571,moving_average,Electronics,Accessories


In [12]:
# Quick sanity checks
print('Unique SKUs:', fc['sku_id'].nunique())
print('Forecast horizon days:', fc['date'].nunique())
print('Methods:', fc['method'].unique())

# Optional: category totals
if 'category' in fc.columns:
    cat_totals = fc.groupby(['date','category'])['y_pred'].sum().reset_index()
    cat_totals.head()


Unique SKUs: 500
Forecast horizon days: 14
Methods: ['moving_average']
